# Injection and labelling

**Goal:** create training labels. We plant fake asteroids whose positions we know (perfect labels), match them to
detections, add real known asteroids from SkyBoT, and call everything else bogus.

Read `docs/injection_labelling.md` alongside this notebook. Training night only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.wcs import WCS

from asthunt.config.load_config import load_config
from asthunt.clean.load_exposure import load_exposure
from asthunt.detect.extract_sources import extract_sources
from asthunt.detect.cut_triplet_stamps import cut_triplet_stamps
from asthunt.simulate.render_point_source import render_point_source
from asthunt.simulate.plan_injections import plan_injections
from asthunt.simulate.inject_point_source import inject_point_source
from asthunt.label.query_known_asteroids import query_known_asteroids
from asthunt.label.label_detections import label_detections

paths = load_config("project")["paths"]
acquire, inject, detect = load_config("acquire"), load_config("inject"), load_config("detect")
rng = np.random.default_rng(load_config("project")["seed"])

night = acquire["nights"]["train"][0]
exposures = [load_exposure(p, acquire["bad_mask_bits"]) for p in sorted((paths["raw"] / "ztf" / night).glob("*sciimg.fits"))]
print(night, len(exposures), "frames")

## 1. The PSF: what a point source looks like on this image
ZTF fits a PSF model for every difference image. Using it makes our fakes look like real point sources *on that night*.

In [ ]:
psf = exposures[0]["psf"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(psf, origin="lower"); axes[0].set_title(f"ZTF PSF model {psf.shape}, sum = {psf.sum():.3f}")
axes[1].imshow(render_point_source(psf, 1000, 0.0, 0.0)[8:17, 8:17], origin="lower"); axes[1].set_title("centred on a pixel")
axes[2].imshow(render_point_source(psf, 1000, 0.5, 0.5)[8:17, 8:17], origin="lower"); axes[2].set_title("on a pixel corner")
plt.show()

## 2. Plan the fakes
300 fake asteroids per night, each moving in a straight line through all 5 frames. Positions use the **real** frame times.

In [ ]:
frame_times = [e["obsjd"] for e in exposures]
plan = plan_injections(inject["asteroids_per_night"], exposures[0]["science"].shape, frame_times,
                       inject["magnitude_range"], inject["rate_range_arcsec_per_hour"],
                       exposures[0]["header"]["PIXSCALE"], inject["edge_margin_pixels"], rng)
print("minutes since first frame:", np.round((np.array(frame_times) - frame_times[0]) * 24 * 60, 1))
plan.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for _, track in plan.groupby("injection_id"):
    ax.plot(track["x"], track["y"], "-", lw=2)
ax.set_title("300 fake tracks (each line = 5 positions)"); ax.set_aspect("equal"); plt.show()

## 3. Inject, detect, label: frame by frame

In [ ]:
labelled_frames = []
injected_images = []
for frame_index, exposure in enumerate(exposures):
    science, difference = exposure["science"].copy(), exposure["difference"].copy()   # keep originals untouched
    this_frame = plan[plan["frame_index"] == frame_index].copy()
    # The pointing shifts between exposures: planned positions are on frame 0's grid,
    # so convert them via the sky (RA, Dec) to this frame's own pixels.
    sky = WCS(exposures[0]["header"]).pixel_to_world(this_frame["x"].to_numpy(), this_frame["y"].to_numpy())
    this_frame["x"], this_frame["y"] = WCS(exposure["header"]).world_to_pixel(sky)
    for _, fake in this_frame.iterrows():
        inject_point_source(science, difference, exposure["psf"], fake["mag"], exposure["magzp"], exposure["gain"],
                            fake["x"], fake["y"], rng)

    detections = extract_sources(difference, exposure["bad"], detect["threshold_sigma"], detect["min_area_pixels"])
    known = query_known_asteroids(exposure["header"], science.shape,
                                   cache_file=paths["interim"] / "known_asteroids" / f"known_asteroids__{exposure['name']}.csv")
    labelled = label_detections(detections, this_frame, known, inject["match_radius_pixels"])
    labelled["frame_index"] = frame_index
    labelled_frames.append(labelled)
    injected_images.append((science, difference))
    print(f"frame {frame_index}: {len(detections)} detections, {(labelled['injection_id'] >= 0).sum()} matched fakes, "
          f"{(labelled['known_asteroid'] != '').sum()} matched known asteroids ({len(known)} known in the field)")

detections = pd.concat(labelled_frames, ignore_index=True)

## 4. Before and after
The same spot, with and without a planted asteroid.

In [ ]:
fake = plan[(plan["frame_index"] == 0) & plan["mag"].between(19, 19.5)].iloc[0]
x, y = fake["x"], fake["y"]
original = exposures[0]
science, difference = injected_images[0]
before = cut_triplet_stamps(original["science"], original["reference"], original["difference"], [x], [y], 31)[0]
after = cut_triplet_stamps(science, science - difference, difference, [x], [y], 31)[0]

fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for row, stamps, name in [(0, before, "before"), (1, after, "after")]:
    for channel, panel in enumerate(["science", "reference", "difference"]):
        axes[row, channel].imshow(stamps[channel], origin="lower", cmap="gray",
                                  vmin=np.percentile(after[channel], 1), vmax=np.percentile(after[channel], 99.5))
        axes[row, channel].set_title(f"{name}: {panel}", fontsize=9); axes[row, channel].axis("off")
plt.suptitle(f"fake asteroid, mag {fake['mag']:.2f}"); plt.show()

## 5. Completeness: what fraction of fakes did we detect?
This curve is the astronomer's standard way to describe a survey's sensitivity.

In [ ]:
found_pairs = detections.loc[detections["injection_id"] >= 0, ["injection_id", "frame_index"]].drop_duplicates()
plan["found"] = plan.set_index(["injection_id", "frame_index"]).index.isin(found_pairs.set_index(["injection_id", "frame_index"]).index)

bins = np.arange(17, 22.25, 0.25)
by_mag = plan.groupby(pd.cut(plan["mag"], bins), observed=True)["found"].mean()
centres = [interval.mid for interval in by_mag.index]
plt.plot(centres, by_mag.values, "o-")
plt.axvline(np.median([e["header"]["MAGLIM"] for e in exposures]), ls="--", c="gray", label="header MAGLIM")
plt.xlabel("injected magnitude"); plt.ylabel("fraction detected (per frame)"); plt.ylim(0, 1); plt.legend(); plt.show()

Two things to notice:
- **Bright fakes top out near 90%, not 100%.** About 10% land on masked pixels (bad columns, saturated stars, unsubtracted areas).
- **The 50% point sits near the header's MAGLIM.** Below it, recovery fades. Pushing that curve fainter is what the ML is for.

In [ ]:
per_asteroid = plan.groupby("injection_id").agg(rate=("rate_arcsec_per_hour", "first"), found=("found", "mean"))
groups = per_asteroid.groupby(pd.cut(per_asteroid["rate"], np.arange(5, 105, 10)), observed=True)["found"]
summary = pd.DataFrame({"asteroids": groups.size(), "completeness": groups.mean()})
# Uncertainty of a fraction measured from n independent asteroids: sqrt(p * (1 - p) / n)
summary["plus_minus"] = np.sqrt(summary["completeness"] * (1 - summary["completeness"]) / summary["asteroids"])
summary.round(2)

The values jump around between roughly 0.6 and 0.8, but each bin only holds ~30 asteroids, so the uncertainty is
about ±0.08. **Most of the jumps are just noise.** Expected physics: in a 30 s exposure an asteroid moving 100 "/hr
travels 0.8", less than one pixel, so speed should barely matter for detection. Always ask "how many samples?"
before reading meaning into a difference.

## 6. Real known asteroids
Fakes are perfect labels, but they're *our* idea of an asteroid. Known asteroids from SkyBoT are the real thing,
just fewer and brighter.

In [ ]:
known_hits = detections[detections["known_asteroid"] != ""]
print(known_hits[["frame_index", "known_asteroid", "x", "y", "snr"]].to_string())

Look closely: low SNR, a different asteroid in nearly every frame. Are these real?
Estimate how many matches we'd get **by pure chance**: a random point lands within 2 px of some detection with probability
(number of detections × circle area) / image area.

In [ ]:
frame0 = detections[detections["frame_index"] == 0]
good_area = (~exposures[0]["bad"]).sum()
chance_per_point = len(frame0) * np.pi * inject["match_radius_pixels"] ** 2 / good_area
known_in_field = len(query_known_asteroids(exposures[0]["header"], exposures[0]["science"].shape,
                                           cache_file=paths["interim"] / "known_asteroids" / f"known_asteroids__{exposures[0]['name']}.csv"))
print(f"chance that a random position matches some detection: {chance_per_point:.2%}")
print(f"expected chance matches per frame: {known_in_field * chance_per_point:.1f}  (we saw 1-2)")

So most "known asteroid" matches are **coincidences with noise**: that's label noise. Only `2003 QW64` appears in two
frames at positions consistent with motion. The fix: only use known asteroids bright enough to be detected
(`known_asteroid_max_v_mag: 20.5` in `configs/inject.yaml`). The same chance-match effect explains why even mag-22 fakes
show a little "completeness": a few are matched to unrelated noise.

#### Update, found later (in the Real/bogus CNN phase)
The CNN gave its highest scores to some detections labelled "bogus". Looking at them: clean new point sources, and **all of them were known asteroids**, some as bright as V = 17. Our 2-px radius had missed them.
- Measured on all nights: SkyBoT's positions sit **~2.9" from our detections, in a constant direction** (scatter only 0.1–0.5 px).
- Our WCS agrees with Gaia stars to **0.04 px**, and **JPL Horizons** agrees with our detections to **0.1–0.3"**. So the offset is in SkyBoT's predictions, not in our pipeline.
- Fix: `known_match_radius_pixels: 4.0` for known asteroids (fakes keep 2 px). Re-labelling added 2–70 real labels per night; the test night went from 12 to 82 known-asteroid detections.

**Lessons:** (1) chance coincidences explained *some* matches, but a systematic error hid behind them. Always check the *direction* of offsets, not just their size. (2) A model's most confident "mistakes" are the best place to look for label errors.


## 7. The label table and the imbalance

In [ ]:
counts = detections["label"].value_counts().rename({0: "bogus", 1: "real"})
print(counts.to_string())
print(f"\nbogus per real detection: {counts['bogus'] / counts['real']:.0f} to 1")

Even with 300 planted asteroids per night, bogus outnumbers real roughly 25 to 1. On the real sky (without fakes) it's
worse: a handful of real asteroids among 40,000 detections. Keep this in mind for the metrics phase.

## Exercises
1. Re-run with `asteroids_per_night = 3000`. What problem appears? (Hint: fakes start landing on each other and on the same detections.)
2. Change `match_radius_pixels` to 5. Does completeness at mag 21.5-22 go up? Why is that misleading?
3. In section 4, try a mag-21.5 fake. Can you see it by eye in the difference stamp?